# The story of the perturbation retrievals...



In [1]:
# conda env 'vis2_clone'
from pathlib import Path
import sys
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr
import seaborn as sns

PROJECT_ROOT = Path().resolve().parent.parent.parent
sys.path.append(str(PROJECT_ROOT))

In [2]:
from src.ExperimentSet import ExperimentSet
from src.Retrieval import Retrieval

In [3]:
map_exp_name = {
    'default': {'name':'Default', 'order':1},
    '0p1': {'name':'Bright Surface', 'order':2},
    '1p33': {'name':'Dark Surface', 'order':3},
    '0p150': {'name':'Low AOD', 'order':4},
    '0p250': {'name':'High AOD', 'order':5},
    'mix22': {'name':'Aerosol Mixture 22', 'order':6},
    'mix54': {'name':'Aerosol Mixture 54', 'order':7},
    'unconstrained': {'name':'Unconstrained', 'order':8},
    'LCL_CTH': {'name':'LCL + CTH', 'order':9},
    'unconstrained_darksurf': {'name':'Unc. Dark Surf.', 'order':10},
    'periodic': {'name':'Periodic', 'order':11},
    'slow': {'name':'Slow Cloud Motion', 'order':12},
    'fast': {'name':'Fast Cloud Motion', 'order':13},
    'slow_unconstrained': {'name':'Slow Cloud Motion Unc.', 'order':14},
    'fast_unconstrained': {'name':'Fast Cloud Motion Unc.', 'order':15}
}

In [4]:
EXPERIMENT_SET = PROJECT_ROOT / 'experiments' / 'CAMP2Ex_RF17' / 'V08'
V08 = ExperimentSet(EXPERIMENT_SET)
V08.set_retrieval_list()
V08.load_retrievals(iteration=-1, extinction_th=[0.3, np.inf], tau_th=[0.3, np.inf])

In [5]:
extinction_all = V08.to_dataframe(0.67, 'cloud', 'extinction', name_map=map_exp_name)
bins = np.arange(0.350, 1.251, 0.1)
extinction_all['altitude_bin'] = pd.cut(extinction_all['z'], bins=bins)
cth_all = V08.to_dataframe(0.67, 'cloud', 'cth', name_map=map_exp_name)
cbh_all = V08.to_dataframe(0.67, 'cloud', 'cbh', name_map=map_exp_name)
geometric_thickness_all = V08.to_dataframe(0.67, 'cloud', 'geometric_thickness', name_map=map_exp_name)

# Distributions

These variables have a distribution over the domain. They could be 3D or 2D.

- Cloud top height
- Cloud base height
- Cloud geometric thickness
- Extinction

In [20]:
# f, ax = plt.subplots(1, 1, figsize=(8, 10))
# sns.boxplot(
#     data=extinction_all.loc[
#         np.logical_or.reduce((
#             extinction_all['name'] == 'Default',
#             extinction_all['name'] == 'LCL + CTH',
#             extinction_all['name'] == 'Unconstrained'
#         ))
#     ], 
#     x='extinction', 
#     y='altitude_bin', 
#     hue='name', 
#     whis=(0, 100), 
#     ax=ax
# )

In [21]:
# f, ax = plt.subplots(1, 1, figsize=(8, 10))
# sns.boxplot(
#     data=extinction_all.loc[
#         np.logical_or.reduce((
#             extinction_all['name'] == 'Periodic',
#             extinction_all['name'] == 'Default'
#         ))
#     ], 
#     x='extinction', 
#     y='altitude_bin', 
#     hue='name', 
#     whis=(0, 100), 
#     ax=ax
# )

In [22]:
# f, ax = plt.subplots(1, 1, figsize=(8, 10))
# sns.boxplot(
#     data=extinction_all.loc[
#         np.logical_or.reduce((
#             extinction_all['name'] == 'Bright Surface',
#             extinction_all['name'] == 'Default',
#             extinction_all['name'] == 'Dark Surface'
#         ))
#     ], 
#     x='extinction', 
#     y='altitude_bin', 
#     hue='name', 
#     whis=(0, 100), 
#     ax=ax
# )

In [23]:
# f, ax = plt.subplots(1, 1, figsize=(8, 10))
# sns.boxplot(
#     data=extinction_all.loc[
#         np.logical_or.reduce((
#             extinction_all['name'] == 'Low AOD',
#             extinction_all['name'] == 'Default',
#             extinction_all['name'] == 'High AOD'
#         ))
#     ], 
#     x='extinction', 
#     y='altitude_bin', 
#     hue='name', 
#     whis=(0, 100), 
#     ax=ax
# )

In [24]:
# f, ax = plt.subplots(1, 1, figsize=(8, 20))
# sns.boxplot(data=extinction_all, x='extinction', y='altitude_bin', hue='name', whis=(0, 100), ax=ax)

In [25]:
# f, ax = plt.subplots(1, 1, figsize=(8, 20))
# sns.boxplot(data=extinction_all, x='extinction', y='name', hue='altitude_bin', whis=(0, 100), ax=ax)

In [ ]:
# # build an order of display names consistent with the existing 'order' column
# order = extinction_all[['name', 'order']].drop_duplicates().sort_values('order')['name'].tolist()
# sns.boxplot(data=extinction_all, x='extinction', y='name', order=order, whis=(0, 100))

In [ ]:
# f, ax = plt.subplots(1, 1, figsize=(8, 20))
# sns.violinplot(data=cth_all, x='cth', y='name', order=order, ax=ax)

In [ ]:
# sns.boxplot(data=cth_all, x='cth', y='name', order=order)

In [ ]:
# sns.boxplot(data=geometric_thickness_all, x='geometric_thickness', y='name', order=order)

# Extrinsic properteis: second set

The second set of extrinsic properties are the extrinsic properties that have a single statistic over the whole cloud domain. These include

- Mean & standard deviation CTH
- Mean & standard deviation CBH
- Mean & standard deviation geometric thickness
- Cloud volume
- Cloud fraction

In [6]:
cth_all.groupby(['name', 'order']).cth.agg(['mean', 'std', 'min', 'max', 'median']).reset_index().sort_values('order')

,name,order,mean,std,min,max,median
4,Default,1,0.691384,0.165211,0.4,1.1,0.6
2,Bright Surface,2,0.742059,0.165707,0.4,1.1,0.7
3,Dark Surface,3,0.669099,0.162221,0.4,1.1,0.6
9,Low AOD,4,0.678387,0.159852,0.4,1.1,0.6
7,High AOD,5,0.697084,0.164688,0.4,1.1,0.6
0,Aerosol Mixture 22,6,0.694598,0.164662,0.4,1.1,0.6
1,Aerosol Mixture 54,7,0.682797,0.159740,0.4,1.1,0.6
14,Unconstrained,8,1.069487,0.345805,0.0,1.4,1.2
8,LCL + CTH,9,0.692928,0.165359,0.4,1.1,0.6
13,Unc. Dark Surf.,10,1.072965,0.340738,0.1,1.4,1.2


In [7]:
cbh_all.groupby(['name', 'order']).cbh.agg(['mean', 'std', 'min', 'max', 'median']).reset_index().sort_values('order')

,name,order,mean,std,min,max,median
4,Default,1,0.527825,0.146442,0.4,1.1,0.5
2,Bright Surface,2,0.580882,0.158400,0.4,1.1,0.6
3,Dark Surface,3,0.519612,0.147439,0.4,1.1,0.5
9,Low AOD,4,0.506607,0.140111,0.4,1.1,0.5
7,High AOD,5,0.541166,0.149524,0.4,1.1,0.5
0,Aerosol Mixture 22,6,0.515097,0.141350,0.4,1.1,0.5
1,Aerosol Mixture 54,7,0.527331,0.146320,0.4,1.1,0.5
14,Unconstrained,8,0.587692,0.370029,0.0,1.4,0.5
8,LCL + CTH,9,0.524894,0.143913,0.4,1.1,0.5
13,Unc. Dark Surf.,10,0.516075,0.384621,0.0,1.4,0.5


In [8]:
geometric_thickness_all.groupby(['name', 'order']).geometric_thickness.agg(['mean', 'std', 'min', 'max', 'median']).reset_index().sort_values('order')

,name,order,mean,std,min,max,median
4,Default,1,0.163559,0.167855,0.0,0.7,0.10
2,Bright Surface,2,0.161176,0.156732,0.0,0.6,0.10
3,Dark Surface,3,0.149487,0.159452,0.0,0.7,0.10
9,Low AOD,4,0.171781,0.162412,0.0,0.7,0.20
7,High AOD,5,0.155918,0.166823,0.0,0.7,0.10
0,Aerosol Mixture 22,6,0.179501,0.167699,0.0,0.7,0.20
1,Aerosol Mixture 54,7,0.155466,0.159692,0.0,0.7,0.10
14,Unconstrained,8,0.481795,0.414502,0.0,1.4,0.35
8,LCL + CTH,9,0.168034,0.170392,0.0,0.7,0.10
13,Unc. Dark Surf.,10,0.556889,0.438042,0.0,1.4,0.40


In [18]:
extinction_all.groupby(['name', 'order']).extinction.agg(['mean', 'std', 'min', 'max', 'median']).reset_index().sort_values('order')

,name,order,mean,std,min,max,median
4,Default,1,8.415283,8.585926,0.300962,68.514220,5.469616
2,Bright Surface,2,7.940410,6.904777,0.304158,39.576348,5.812147
3,Dark Surface,3,9.082452,10.772338,0.300121,124.476958,5.051927
9,Low AOD,4,7.519411,8.527933,0.302214,81.503991,4.399314
7,High AOD,5,9.393918,9.008950,0.316198,63.833102,6.548327
0,Aerosol Mixture 22,6,7.498146,7.273117,0.300439,51.634662,4.996583
1,Aerosol Mixture 54,7,9.602847,10.675483,0.304469,86.328630,5.788028
14,Unconstrained,8,6.040523,6.646254,0.303711,69.897971,3.797167
8,LCL + CTH,9,8.362713,8.513315,0.308571,66.645166,5.394752
13,Unc. Dark Surf.,10,5.323646,6.124112,0.300040,68.658084,3.146613


In [9]:
brf_ths = {
    'DF': 0.120,
    'CF': 0.076,
    'BF': 0.054,
    'AF': 0.044,
    'AN': 0.040,
    'AA': 0.045,
    'BA': 0.052,
    'CA': 0.068,
    'DA': 0.092
}
for rpath in V08.retrievals:
    r = V08.retrievals[rpath]
    r.add_threshold_mask(r.obs, 'MISR', 'threshold_mask', 'brf', brf_ths)

In [10]:
def map_df(df, map_exp_name, ES):
    df = df.copy(deep=True)
    df['Retrieval Name'] = ''
    df['Order'] = 0
    for i in range(len(df)):
        new_name, order = map_exp_name[ES.retrievals[Path(df.reset_index().loc[i, 'retrieval'])].exp_name]
        df.loc[df.index == df.reset_index().loc[i, 'retrieval'], 'Retrieval Name'] = new_name
        df.loc[df.index == df.reset_index().loc[i, 'retrieval'], 'Order'] = order
    return df.reset_index().set_index('Order')

In [11]:
map_exp_name = {
    'default': ('Default', 1),
    '0p1': ('Bright Surface', 2),
    '1p33': ('Dark Surface', 3),
    '0p150': ('Low AOD', 4),
    '0p250': ('High AOD', 5),
    'mix22': ('Aerosol Mixture 22', 6),
    'mix54': ('Aerosol Mixture 54', 7),
    'unconstrained': ('Unconstrained', 8),
    'LCL_CTH': ('LCL + CTH', 9),
    'unconstrained_darksurf': ('Unc. Dark Surf.', 10),
    'periodic': ('Periodic', 11),
    'slow': ('Slow Cloud Motion', 12),
    'fast': ('Fast Cloud Motion', 13),
    'slow_unconstrained': ('Slow Cloud Motion Unc.', 14),
    'fast_unconstrained': ('Fast Cloud Motion Unc.', 15)
}

In [12]:
V08_all = ExperimentSet(EXPERIMENT_SET)
V08_all.set_retrieval_list()
V08_all.load_retrievals(iteration=-1, extinction_th=[0.0, np.inf], tau_th=[0.0, np.inf])

In [13]:
def misr_S_1(y):
    return np.diag(1 / (0.04 * y)**2)
mask_conds = {
    # 'tau0.3mask': (1, 1),
    'threshold_mask': (1, 1),
    'ray_containment': (2, 2)
}
chi2 = V08.chi2(misr_S_1, variable='I', mask_conds=mask_conds)

stats_th = V08.cloud_stats(0.67, 'cloud')
stats_all = V08_all.cloud_stats(0.67, 'cloud')
stats_diff = stats_all - stats_th

chi2['Masked Cloud Volume'] = stats_diff['volume']
chi2['Unmasked Cloud Volume'] = stats_th['volume']

ref_key = Path('/data/gdi/b/johntl3/LundstromPub2025/experiments/CAMP2Ex_RF17/V08/default')
stats_th_ref = V08.compare_to(ref_key, 0.67, 'cloud', extinction_th=[0.3, np.inf], tau_th=[0.3, np.inf])

stats_th['Relative Bias (%)'] = stats_th_ref['relative bias']
stats_th['Relative Difference (%)'] = stats_th_ref['relative difference']

stats_th['Default CF'] = float(stats_th.loc[stats_th.index==str(ref_key), 'cloud fraction'])
stats_th['Default tau bar'] = float(stats_th.loc[stats_th.index==str(ref_key), 'tau bar'])
stats_th['Default volume'] = float(stats_th.loc[stats_th.index==str(ref_key), 'volume'])
stats_th['Rel CF Diff'] = 100*(stats_th['cloud fraction'] - stats_th['Default CF']) / stats_th['Default CF']
stats_th['Rel tau bar Diff'] = 100*(stats_th['tau bar'] - stats_th['Default tau bar']) / stats_th['Default tau bar']
stats_th['Rel volume Diff'] = 100*(stats_th['volume'] - stats_th['Default volume']) / stats_th['Default volume']


/tmp/ipykernel_510688/2645029070.py:23: FutureWarning: Calling float on a single element Series is deprecated and will raise a TypeError in the future. Use float(ser.iloc[0]) instead
  stats_th['Default CF'] = float(stats_th.loc[stats_th.index==str(ref_key), 'cloud fraction'])
/tmp/ipykernel_510688/2645029070.py:24: FutureWarning: Calling float on a single element Series is deprecated and will raise a TypeError in the future. Use float(ser.iloc[0]) instead
  stats_th['Default tau bar'] = float(stats_th.loc[stats_th.index==str(ref_key), 'tau bar'])
/tmp/ipykernel_510688/2645029070.py:25: FutureWarning: Calling float on a single element Series is deprecated and will raise a TypeError in the future. Use float(ser.iloc[0]) instead
  stats_th['Default volume'] = float(stats_th.loc[stats_th.index==str(ref_key), 'volume'])


In [15]:
map_df(chi2, map_exp_name, V08).loc[:, ['Retrieval Name', 'chi2', 'Masked Cloud Volume', 'Unmasked Cloud Volume']]

,Retrieval Name,chi2,Masked Cloud Volume,Unmasked Cloud Volume
Order,,,,
9,LCL + CTH,10.871575,5.082,13.211687500000004
8,Unconstrained,12.489859,10.050562,20.588906250000008
10,Unc. Dark Surf.,10.372845,14.758219,28.824468750000012
11,Periodic,20.778908,9.460688,12.534843750000006
4,Low AOD,8.661792,12.213438,16.289625000000004
5,High AOD,17.347160,3.198937,10.655562500000004
6,Aerosol Mixture 22,10.395386,5.65675,14.4821875
7,Aerosol Mixture 54,11.470972,5.687,10.776562500000004
13,Fast Cloud Motion,13.047756,5.709688,10.980750000000002


In [17]:
map_df(stats_th, map_exp_name, V08).loc[:, ['Retrieval Name', 'Relative Bias (%)', 'Relative Difference (%)', 'Default CF', 'cloud fraction', 'Rel CF Diff', 'Default tau bar', 'tau bar', 'Rel tau bar Diff', 'Default volume', 'volume', 'Rel volume Diff']]

,Retrieval Name,Relative Bias (%),Relative Difference (%),Default CF,cloud fraction,Rel CF Diff,Default tau bar,tau bar,Rel tau bar Diff,Default volume,volume,Rel volume Diff
Order,,,,,,,,,,,,
9,LCL + CTH,0.123786,0.338949,0.167574,0.16733727810650886,-0.141243,2.061031,2.066429954063104,0.261946,13.113375,13.211687500000004,0.749712
8,Unconstrained,20.978603,3.069472,0.167574,0.18461538461538463,10.169492,2.061031,2.060186616551134,-0.040977,13.113375,20.588906250000008,57.00692
10,Unc. Dark Surf.,48.698434,3.197021,0.167574,0.22674556213017752,35.310734,2.061031,2.075117373971214,0.683455,13.113375,28.824468750000012,119.809689
11,Periodic,-36.299753,2.267616,0.167574,0.15550295857988167,-7.20339,2.061031,1.4157179267412454,-31.310212,13.113375,12.534843750000006,-4.411765
4,Low AOD,10.964002,0.566757,0.167574,0.21136094674556213,26.129944,2.061031,1.8137525607094735,-11.99781,13.113375,16.289625000000004,24.221453
5,High AOD,-9.238868,0.507701,0.167574,0.13798816568047337,-17.655367,2.061031,2.2703312311641217,10.155114,13.113375,10.655562500000004,-18.742791
6,Aerosol Mixture 22,-1.599205,0.779176,0.167574,0.17088757396449705,1.977401,2.061031,1.9887740882413436,-3.50587,13.113375,14.4821875,10.438293
7,Aerosol Mixture 54,-6.227550,1.098185,0.167574,0.1472189349112426,-12.146893,2.061031,2.2000090934104177,6.743126,13.113375,10.776562500000004,-17.820069
13,Fast Cloud Motion,-2.108493,3.451359,0.167574,0.16662721893491125,-0.564972,2.061031,2.028654334351273,-1.570904,13.113375,10.980750000000002,-16.262976
